In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):                       # простая обёртка: принимает строку, возвращает ответ LLM
    response = openai_client.responses.create(
        model='gpt-5.4-mini',          # лёгкая и дешёвая модель для экспериментов
        input=prompt                   # передаём промпт напрямую как строку (без ролей)
    )
    return response.output_text        # возвращаем только текст ответа

In [4]:
question = 'I just discovered the course. Can I join now?'  # тестовый вопрос без контекста
answer = llm(question)  # вызов LLM без контекста → модель может галлюцинировать (нет данных о конкретном курсе)
print(answer)

Yes — in most cases you can join now, but it depends on whether the course is still open for enrollment.

If you want, I can help you draft a quick message to the instructor or course admin like:

> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll now?

If you tell me the course name or platform, I can make the message more specific.


In [5]:
# ручной контекст из нескольких FAQ-записей — заменяет поиск, демонстрирует принцип RAG
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don't post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [6]:
# вручную собранный промпт: инструкция + вопрос + контекст (до рефакторинга в функции)
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [7]:
question = 'I just discovered the course. Can I join now?'  # тот же вопрос, но теперь с контекстом
answer = llm(prompt)   # LLM отвечает по контексту → точный ответ вместо галлюцинации
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [8]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [9]:
documents = []                   # итоговый список всех FAQ-документов со всех курсов
url_prefix = 'https://datatalks.club/faq'  # базовый URL — к нему добавляем path каждого курса

for course in courses_raw:       # перебираем каждый курс (mlops, llm, de-zoomcamp, ...)
    course_url = f'{url_prefix}{course['path']}'  # формируем полный URL FAQ этого курса

    course_response = requests.get(course_url)    # скачиваем FAQ курса
    course_response.raise_for_status()            # бросает HTTPError при статусе 4xx/5xx
    course_data = course_response.json()          # парсим JSON в список документов курса

    documents.extend(course_data)  # добавляем документы курса в общий список

len(documents)  # выводим итоговое количество (должно быть ~1154)

1342

In [10]:
documents[1100]  # просматриваем произвольный документ — проверяем структуру {id, course, section, question, answer}

{'id': 'f580e98fdc',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Waitress on Windows / Git Bash: "waitress-serve: command not found"',
 'answer': '`pip install waitress` does install `waitress-serve.exe`, but Python\'s `Scripts/` directory may not be on Git Bash\'s `PATH`. The pip output usually warns about this:\n\n```\nWARNING: The script waitress-serve.exe is installed in \'C:\\Users\\<you>\\...\\Scripts\'\nwhich is not on PATH.\n```\n\nTo fix, add that `Scripts` directory to Git Bash\'s `PATH` permanently:\n\n```bash\nnano ~/.bashrc\n# add this line, with the path from the warning:\nexport PATH="/c/Users/<you>/.../Scripts:$PATH"\n```\n\nClose Git Bash and reopen it. `waitress-serve --help` should now work.\n\nIf you\'re using `uv`, this isn\'t an issue because `uv run waitress-serve ...` runs the binary directly from the venv without needing it on `PATH`.'}

In [11]:
from minsearch import Index  # импортируем in-memory поисковый движок

index = Index(
    text_fields=['question', 'section', 'answer'],  # по этим полям ведётся полнотекстовый TF-IDF поиск
    keyword_fields=['course']                        # это поле используется только для фильтрации
)
index.fit(documents)  # строим индекс по всем 1154 документам (быстро, всё в памяти)


In [12]:
search_results = index.search(
    question,                                      # поисковый запрос пользователя
    boost_dict={'question': 2.0, 'section': 0.5}, # вопрос важнее раздела при ранжировании
    filter_dict={'course': 'llm-zoomcamp'},         # ищем только в документах LLM Zoomcamp
    num_results=5                                  # возвращаем топ-5 наиболее релевантных
)

search_results  # выводим найденные документы для проверки

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [13]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [14]:
results = index.search(
    question,
    num_results=5,
    boost_dict={'question': 2.0, 'section': 0.5}
)

In [15]:
[doc['question'] for doc in results]

['I just discovered the course. Can I still join?',
 'The course has already started. Can I still join it?',
 'Course - Can I still join the course after the start date?',
 'Course: Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?']

In [16]:
results = index.search(
    question,
    num_results=5,
    boost_dict={'question': 0.5, 'section': 2}
)

In [17]:
[doc['question'] for doc in results]

['I just discovered the course. Can I still join?',
 'The course has already started. Can I still join it?',
 'Course: When does the course start?',
 'I’m new to Slack and can’t find the course channel. Where is it?',
 'How do I join Slack if the invite email didn’t arrive?']

In [18]:
def search(question, course='llm-zoomcamp'):   # обёртка поиска с дефолтным курсом
    boost_dict = {'question': 2.0, 'section': 0.5}  # веса полей при ранжировании
    filter_dict = {'course': course}                 # фильтрация по курсу

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5         # всегда топ-5
    )

In [19]:
search_results = search(question)  # выполняем поиск через новую функцию-обёртку
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [20]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''  # системный промпт: будет передан в роли 'developer', задаёт правила поведения ассистента

In [21]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''  # шаблон пользовательского сообщения: {question} и {context} заменяются через .format() (опечатка в имени)

In [22]:
lines = []  # список строк, из которых собирается текстовый контекст

for doc in search_results:                 # для каждого найденного документа
    lines.append(doc['section'])            # раздел FAQ (например, "Module 1: Intro")
    lines.append('Q: ' + doc['question'])   # вопрос из FAQ
    lines.append('A: ' + doc['answer'])     # ответ из FAQ
    lines.append('')                        # пустая строка — разделитель между документами

print('\n'.join(lines).strip())  # объединяем строки в текст, убираем пробелы по краям

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [23]:
def build_context(search_results):
    lines = []  # список строк, из которых собирается текстовый контекст

    for doc in search_results:                 # для каждого найденного документа
        lines.append(doc['section'])            # раздел FAQ (например, "Module 1: Intro")
        lines.append('Q: ' + doc['question'])   # вопрос из FAQ
        lines.append('A: ' + doc['answer'])     # ответ из FAQ
        lines.append('')                        # пустая строка — разделитель между документами

    return '\n'.join(lines).strip()  # объединяем строки в текст, убираем пробелы по краям

In [24]:
context = build_context(search_results)  # форматируем список документов в читаемый текст
prompt = USER_PROMPT_TEMPALATE.format(
        question=question,  # подставляем вопрос пользователя
        context=context     # подставляем найденный контекст
    )
print(prompt)


Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your projec

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)  # форматируем список документов в читаемый текст
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,  # подставляем вопрос пользователя
        context=context     # подставляем найденный контекст
    )
    return prompt.strip()  # убираем лишние пробелы по краям готового промпта

In [26]:
prompt = build_prompt(question, search_results)  # строим готовый промпт для передачи в LLM

In [27]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',  # быстрая и дешёвая модель
    input=prompt           # промпт как одна строка (без разделения на роли)
)

In [28]:
response

Response(id='resp_0cc6c05d7990f507006a2bfebb22c0819194e4d8b3fee9422c', created_at=1781268155.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0cc6c05d7990f507006a2bfebb616c819194a82789f84b0415', content=[ResponseOutputText(annotations=[], text='Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781268156.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(effort='none', generate_summary=None, summary=None, context='current_turn'), safety

In [29]:
response.output_text  # удобное свойство: возвращает текст первого ответа модели

'Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [30]:
response.output[0]

ResponseOutputMessage(id='msg_0cc6c05d7990f507006a2bfebb616c819194a82789f84b0415', content=[ResponseOutputText(annotations=[], text='Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [31]:
response.output[0].content[0]

ResponseOutputText(annotations=[], text='Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])

In [32]:
response.output[0].content[0].text  # то же самое через полную иерархию объектов ответа

'Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [33]:
response.usage  # статистика токенов: input_tokens, output_tokens, total_tokens

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=38, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=372)